# Open-Domain Claim Verification — Step 3 Only

Loads pre-computed evidence from `reference/evidence/` and runs only **Step 3** (DeBERTa-v3-large NLI verdict prediction) on four datasets: SciFact, PubMedQA, HealthFC, CoVERT. Steps 1–2 (retrieval and evidence selection) are skipped. Results are saved to `reproduction/step3-only/results/`.

| Knowledge source | Retrieval method | Evidence folder |
|---|---|---|
| Wikipedia | BM25 | `reference/evidence/Wikipedia/BM25 Search/` |
| Wikipedia | Semantic (BioSimCSE) | `reference/evidence/Wikipedia/Semantic Search/` |
| PubMed    | BM25 | `reference/evidence/PubMed/BM25 Search/` |
| PubMed    | Semantic (BioSimCSE) | `reference/evidence/PubMed/Semantic Search/` |
| Google    | Web search snippets | `reference/evidence/Google/` |

In [ ]:
# If running locally install manually:
#   pip install transformers torch scikit-learn pandas numpy openpyxl
!pip install transformers torch scikit-learn pandas numpy openpyxl --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Running on Colab - Drive mounted.')
except ImportError:
    print('Running locally - skipping Drive mount.')


In [ ]:
import os, csv
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

try:
    import google.colab
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

BASE_PATH = '/content/drive/MyDrive/NLP-Group-Project' if RUNNING_ON_COLAB else '../..'

DATASETS_PATH         = os.path.join(BASE_PATH, 'data')
ORIGINAL_RESULTS_PATH = os.path.join(BASE_PATH, 'reference', 'evidence')
RESULTS_PATH          = os.path.join(BASE_PATH, 'reproduction', 'step3-only', 'results')
os.makedirs(RESULTS_PATH, exist_ok=True)

DATASETS_TO_RUN = ['scifact', 'pubmedqa', 'healthfc', 'covert']
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Running on Colab: {RUNNING_ON_COLAB}')
print(f'Using device:     {device}')
print(f'Datasets:         {os.path.abspath(DATASETS_PATH)}')
print(f'Evidence:         {os.path.abspath(ORIGINAL_RESULTS_PATH)}')
print(f'Saving to:        {os.path.abspath(RESULTS_PATH)}')

## Step 0 — Load Datasets

Labels normalised to **1 = SUPPORTED**, **0 = REFUTED**. NEI claims excluded.

| Dataset | Label fix |
|---|---|
| HealthFC | `0=SUPPORTED, 1=NEI, 2=REFUTED` in CSV → remapped `0→1`, `2→0` |
| PubMedQA | `'maybe'` (NEI) excluded explicitly via `.isin(['yes','no'])` |
| CoVERT   | `SUPPORTS→1`, `REFUTES→0` |
| SciFact  | Already binary, no change needed |


In [ ]:
scifact_df     = pd.read_csv(os.path.join(DATASETS_PATH, 'scifact_no-nei_dataset.csv'), index_col=[0])
scifact_claims = scifact_df.claim.tolist()
scifact_labels = scifact_df.label.tolist()

pubmedqa_df     = pd.read_json(os.path.join(DATASETS_PATH, 'pubmedqa.json')).transpose()
pubmedqa_df     = pubmedqa_df[pubmedqa_df.final_decision.isin(['yes', 'no'])]
pubmedqa_claims = pubmedqa_df.QUESTION.tolist()
pubmedqa_labels = [1 if l == 'yes' else 0 for l in pubmedqa_df.final_decision.tolist()]

healthfc_df     = pd.read_csv(os.path.join(DATASETS_PATH, 'healthFC_annotated.csv'))
healthfc_df     = healthfc_df[healthfc_df.label != 1].copy()
healthfc_claims = healthfc_df.en_claim.tolist()
healthfc_labels = [1 if l == 0 else 0 for l in healthfc_df.label.tolist()]

covert_df     = pd.read_json(os.path.join(DATASETS_PATH, 'CoVERT_FC_annotations.jsonl'), lines=True)
covert_df     = covert_df[covert_df.label != 'NOT ENOUGH INFO'].copy()
covert_df['claim'] = covert_df.claim.str.replace('@username', '', regex=False).str.replace('\n', ' ', regex=False).str.strip()
covert_claims = covert_df.claim.tolist()
covert_labels = [1 if l == 'SUPPORTS' else 0 for l in covert_df.label.tolist()]

all_datasets = {
    'scifact':  (scifact_claims,  scifact_labels),
    'pubmedqa': (pubmedqa_claims, pubmedqa_labels),
    'healthfc': (healthfc_claims, healthfc_labels),
    'covert':   (covert_claims,   covert_labels),
}
gold_labels = {ds: labels for ds, (_, labels) in all_datasets.items()}

print('Dataset sizes (after excluding NEI):')
for name, (claims, labels) in all_datasets.items():
    print(f'  {name:<12}: {len(claims):>4} claims | {sum(labels)} supported, {len(labels)-sum(labels)} refuted')


## Step 3 — NLI Model Setup

Loads DeBERTa-v3-large fine-tuned on MNLI/FEVER/ANLI. Predicts SUPPORTED if P(entailment) > P(contradiction).


In [ ]:
NLI_MODEL_NAME = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli'
print(f'Loading: {NLI_MODEL_NAME}')
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME, model_max_length=1024)
nli_model     = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
print('NLI model loaded.')


In [ ]:
class ClaimDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels
    def __getitem__(self, idx):
        item = {k: v[idx].clone().detach() for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)


def predict_verdicts(joint_list, batch_size=16):
    encodings = nli_tokenizer(
        joint_list, return_tensors='pt', truncation='only_first',
        padding=True, max_length=1024
    )
    dataset = ClaimDataset(encodings, [0] * len(joint_list))
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    nli_model.eval()
    nli_model.to(device)
    preds = []
    with torch.no_grad():
        for batch in loader:
            logits = nli_model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device)
            )[0]
            probs = logits.softmax(dim=1)
            preds.extend((probs[:, 0] > probs[:, 2]).cpu().numpy().astype(int).tolist())
    return np.array(preds)


def run_and_save(joint_lists, label_source, save_subdir, label):
    save_dir = os.path.join(RESULTS_PATH, save_subdir)
    os.makedirs(save_dir, exist_ok=True)
    results = {}
    print(f'\n--- {label} ---')
    print(f'{"Dataset":<12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Accuracy":>10}')
    print('-' * 55)
    for ds in DATASETS_TO_RUN:
        gold  = label_source[ds]
        preds = predict_verdicts(joint_lists[ds])
        p = precision_score(gold, preds, average='binary', zero_division=0)
        r = recall_score(   gold, preds, average='binary', zero_division=0)
        f = f1_score(       gold, preds, average='binary', zero_division=0)
        a = accuracy_score( gold, preds)
        results[ds] = {'precision': p, 'recall': r, 'f1': f, 'accuracy': a}
        print(f'{ds:<12} {p*100:>10.1f} {r*100:>10.1f} {f*100:>10.1f} {a*100:>10.1f}')
    print('-' * 55)
    with open(os.path.join(save_dir, 'metrics.csv'), 'w', newline='', encoding='utf-8') as fh:
        w = csv.DictWriter(fh, fieldnames=['dataset','precision','recall','f1','accuracy'])
        w.writeheader()
        for ds, s in results.items():
            w.writerow({'dataset': ds, 'precision': round(s['precision']*100,1),
                        'recall': round(s['recall']*100,1), 'f1': round(s['f1']*100,1),
                        'accuracy': round(s['accuracy']*100,1)})
    print(f'Saved: {os.path.join(save_dir, "metrics.csv")}')
    return results


def read_bm25_file(filepath):
    with open(filepath, encoding='utf-8') as f:
        raw = f.read().splitlines()
    entries, current = [], None
    for line in raw:
        if '[SEP]' in line:
            if current is not None:
                entries.append(current)
            current = line
        elif current is not None:
            current = current + ' ' + line.strip()
    if current is not None:
        entries.append(current)
    return entries


def parse_google_file(filepath):
    with open(filepath, encoding='utf-8') as f:
        content = f.read()
    joint = []
    for block in content.split('\n\n'):
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        if not lines or lines[0] != 'CLAIM':
            continue
        claim = lines[1] if len(lines) > 1 else ''
        try:
            ev_idx = lines.index('EVIDENCE')
        except ValueError:
            joint.append(f'{claim} [SEP] ')
            continue
        ev_lines = lines[ev_idx + 1:]
        snippets = [ev_lines[i] for i in range(2, len(ev_lines), 3)
                    if ev_lines[i] not in ('No results found!', '')]
        joint.append(f'{claim} [SEP] {" ".join(snippets)}')
    return joint


def align_entries_to_claims(entries, claims):
    claim_to_entry = {e.split(' [SEP] ')[0].strip(): e for e in entries}
    missing = [c for c in claims if c not in claim_to_entry]
    if missing:
        print(f'  WARNING: {len(missing)} claim(s) have no entry — using empty evidence:')
        for c in missing:
            print(f'    - {c[:100]}')
    return [claim_to_entry.get(c, f'{c} [SEP] ') for c in claims]


def load_source(base_path, files_dict, parser=read_bm25_file):
    joint = {}
    for ds in DATASETS_TO_RUN:
        claims, _ = all_datasets[ds]
        entries = parser(os.path.join(base_path, files_dict[ds]))
        if len(entries) != len(claims):
            entries = align_entries_to_claims(entries, claims)
        assert len(entries) == len(claims), f'{ds}: {len(entries)} entries vs {len(claims)} claims'
        joint[ds] = entries
        print(f'{ds}: {len(entries)} entries loaded')
    return joint


def get_f1(results, ds):
    return round(results.get(ds, {}).get('f1', float('nan')) * 100, 1)

def get_acc(results, ds):
    return round(results.get(ds, {}).get('accuracy', float('nan')) * 100, 1)


print('Helper functions defined.')

---
## Wikipedia BM25


In [ ]:
wiki_results = run_and_save(
    load_source(
        os.path.join(ORIGINAL_RESULTS_PATH, 'Wikipedia', 'BM25 Search'),
        {'scifact': 'bm25wiki_lines_scifact.txt', 'pubmedqa': 'bm25wiki_lines_pubmedqa.txt',
         'healthfc': 'bm25wiki_lines_healthfc.txt', 'covert': 'bm25wiki_lines_covert.txt'}),
    gold_labels, 'Wikipedia BM25', 'Wikipedia BM25')

---
## Wikipedia Semantic Search

In [ ]:
wiki_sem_results = run_and_save(
    load_source(
        os.path.join(ORIGINAL_RESULTS_PATH, 'Wikipedia', 'Semantic Search'),
        {'scifact': 'wiki_lines_scifact.txt', 'pubmedqa': 'wiki_lines_pubmedqa.txt',
         'healthfc': 'wiki_lines_healthfc.txt', 'covert': 'wiki_lines_covert.txt'}),
    gold_labels, 'Wikipedia Semantic', 'Wikipedia Semantic')

---
## PubMed BM25


In [ ]:
pubmed_results = run_and_save(
    load_source(
        os.path.join(ORIGINAL_RESULTS_PATH, 'PubMed', 'BM25 Search'),
        {'scifact': 'bm25pubmed_lines_scifact.txt', 'pubmedqa': 'bm25pubmed_lines_pubmedqa.txt',
         'healthfc': 'bm25pubmed_lines_healthfc.txt', 'covert': 'bm25pubmed_lines_covert.txt'}),
    gold_labels, 'PubMed BM25', 'PubMed BM25')

---
## PubMed Semantic Search

Note: `pubmedqa_semantic_lines.txt` and `covert_semantic_lines.txt` are mislabelled in the original results — their contents are swapped. The file mapping below corrects for this.

In [ ]:
# pubmedqa and covert files are swapped in the original results — corrected here
pubmed_sem_results = run_and_save(
    load_source(
        os.path.join(ORIGINAL_RESULTS_PATH, 'PubMed', 'Semantic Search'),
        {'scifact':  'scifact_semantic_lines.txt',
         'pubmedqa': 'covert_semantic_lines.txt',    # swapped in original files
         'healthfc': 'healthfc_semantic_lines.txt',
         'covert':   'pubmedqa_semantic_lines.txt'}),  # swapped in original files
    gold_labels, 'PubMed Semantic', 'PubMed Semantic')

---
## Google

Parses `CLAIM / EVIDENCE / title / url / snippet` blocks. Snippets concatenated directly as evidence (no SPICED step — matches paper).


In [ ]:
google_results = run_and_save(
    load_source(
        os.path.join(ORIGINAL_RESULTS_PATH, 'Google'),
        {'scifact': 'scifact_google_lines.txt', 'pubmedqa': 'pubmedqa_google_lines.txt',
         'healthfc': 'healthfc_google_lines.txt', 'covert': 'covert_google_lines.txt'},
        parser=parse_google_file),
    gold_labels, 'Google', 'Google')

---
## Summary — All Sources vs. Paper


In [ ]:
# F1 scores from Tables 3 & 4 of Vladika & Matthes (EACL 2024)
paper_wiki_bm25   = {'scifact': 74.8, 'pubmedqa': 73.1, 'healthfc': 73.1, 'covert': 75.2}
paper_wiki_sem    = {'scifact': 75.4, 'pubmedqa': 73.2, 'healthfc': 76.5, 'covert': 82.5}
paper_pubmed_bm25 = {'scifact': 76.1, 'pubmedqa': 70.3, 'healthfc': 69.7, 'covert': 79.5}
paper_pubmed_sem  = {'scifact': 76.8, 'pubmedqa': 74.5, 'healthfc': 72.0, 'covert': 76.2}
paper_google      = {'scifact': 82.7, 'pubmedqa': 78.5, 'healthfc': 74.5, 'covert': 72.3}

# --- F1 comparison DataFrame ---
f1_rows = {ds: {
    ('Wikipedia',  'BM25 Paper'):      paper_wiki_bm25[ds],
    ('Wikipedia',  'BM25 Ours'):       get_f1(wiki_results, ds),
    ('Wikipedia',  'Semantic Paper'):  paper_wiki_sem[ds],
    ('Wikipedia',  'Semantic Ours'):   get_f1(wiki_sem_results, ds),
    ('PubMed',     'BM25 Paper'):      paper_pubmed_bm25[ds],
    ('PubMed',     'BM25 Ours'):       get_f1(pubmed_results, ds),
    ('PubMed',     'Semantic Paper'):  paper_pubmed_sem[ds],
    ('PubMed',     'Semantic Ours'):   get_f1(pubmed_sem_results, ds),
    ('Google',     'Paper'):           paper_google[ds],
    ('Google',     'Ours'):            get_f1(google_results, ds),
} for ds in DATASETS_TO_RUN}

f1_df = pd.DataFrame(f1_rows).T
f1_df.index.name = 'Dataset'
f1_df.columns = pd.MultiIndex.from_tuples(f1_df.columns)
f1_df.loc['Average'] = f1_df.mean().round(1)

print('F1 Scores (%)')
display(f1_df)

f1_path = os.path.join(RESULTS_PATH, 'f1_comparison.xlsx')
f1_df.to_excel(f1_path)
print(f'Saved: {f1_path}')

# --- Accuracy DataFrame (ours only — paper does not report accuracy) ---
acc_rows = {ds: {
    ('Wikipedia', 'BM25'):      get_acc(wiki_results, ds),
    ('Wikipedia', 'Semantic'):  get_acc(wiki_sem_results, ds),
    ('PubMed',    'BM25'):      get_acc(pubmed_results, ds),
    ('PubMed',    'Semantic'):  get_acc(pubmed_sem_results, ds),
    ('Google',    'Ours'):      get_acc(google_results, ds),
} for ds in DATASETS_TO_RUN}

acc_df = pd.DataFrame(acc_rows).T
acc_df.index.name = 'Dataset'
acc_df.columns = pd.MultiIndex.from_tuples(acc_df.columns)
acc_df.loc['Average'] = acc_df.mean().round(1)

print('\nAccuracy (%) — paper does not report this metric')
display(acc_df)

acc_path = os.path.join(RESULTS_PATH, 'accuracy_comparison.xlsx')
acc_df.to_excel(acc_path)
print(f'Saved: {acc_path}')

In [ ]:
PREV_RESULTS_PATH = os.path.join(BASE_PATH, 'reproduction', 'api-pipeline', 'results')

# Discover available subfolders
available = [d for d in os.listdir(PREV_RESULTS_PATH)
             if os.path.isfile(os.path.join(PREV_RESULTS_PATH, d, 'metrics.csv'))]
print(f'Found previous result folders: {available}')

# Load all metrics.csv files from the api-pipeline results folder
prev = {}
for folder in available:
    df = pd.read_csv(os.path.join(PREV_RESULTS_PATH, folder, 'metrics.csv'))
    df = df.set_index('dataset')
    prev[folder] = df
    print(f'  {folder}: {df.index.tolist()}')

# Current step-3-only F1 results for Wikipedia and PubMed
current_f1 = {
    'Wikipedia BM25':      {ds: get_f1(wiki_results, ds)       for ds in DATASETS_TO_RUN},
    'Wikipedia Semantic':  {ds: get_f1(wiki_sem_results, ds)   for ds in DATASETS_TO_RUN},
    'PubMed BM25':         {ds: get_f1(pubmed_results, ds)     for ds in DATASETS_TO_RUN},
    'PubMed Semantic':     {ds: get_f1(pubmed_sem_results, ds) for ds in DATASETS_TO_RUN},
}

# Build comparison: api-pipeline F1 vs step-3-only F1
rows = {}
for folder, df in prev.items():
    if 'f1' not in df.columns:
        continue
    for ds in DATASETS_TO_RUN:
        if ds not in df.index:
            continue
        rows.setdefault(ds, {})
        rows[ds][(folder, 'API Pipeline F1')] = round(df.loc[ds, 'f1'], 1)
        if folder in current_f1:
            rows[ds][(folder, 'Step3-Only F1')] = current_f1[folder].get(ds, float('nan'))

if rows:
    comp_df = pd.DataFrame(rows).T
    comp_df.index.name = 'Dataset'
    comp_df.columns = pd.MultiIndex.from_tuples(comp_df.columns)
    comp_df.loc['Average'] = comp_df.mean().round(1)
    print('\nF1 comparison — API pipeline vs Step-3-only (this notebook)')
    display(comp_df)

    comp_path = os.path.join(RESULTS_PATH, 'f1_api_vs_step3.xlsx')
    comp_df.to_excel(comp_path)
    print(f'Saved: {comp_path}')
else:
    print('No matching f1 columns found — check the metrics.csv column names in reproduction/api-pipeline/results/')